# நுணரிவு AI · Nunarivu AI
## Offline Tamil AI Tutor for Sri Lankan Students

> **நுணரிவு** (Nunarivu) means *"intelligence / wisdom"* in Tamil

---

**Gemma 4 Good Hackathon** · Tracks: **Future of Education** · **Digital Equity & Inclusivity** · **LiteRT Prize**

**GitHub**: https://github.com/sanjeevakumar610/Nunarivu-AI

---

## What This Notebook Does

This notebook demonstrates the **complete AI pipeline** running inside the Nunarivu AI Android app — the same logic, the same prompt, the same OCR parameters — just on Kaggle's GPU instead of a student's phone.

| Step | What you'll see |
|------|-----------------|
| 1 | A real Sri Lankan Tamil government textbook page (image) |
| 2 | Why standard PDF extraction returns **zero usable text** from these books |
| 3 | Tesseract OCR extracting real Tamil Unicode text from the same page |
| 4 | **Gemma 4** answering a Tamil question grounded strictly in that page |
| 5 | You change the question — Gemma 4 answers it |

---

## The Problem

**542,344 Tamil-medium students** in Sri Lanka study from government textbooks and have no AI learning tools they can actually use:

- **No internet** in rural and estate areas — some areas have zero connectivity
- **Cannot afford** tutoring, subscriptions, or even mobile data (~$95/month household income)
- **No Tamil AI tools work offline** — every existing app requires internet or a paid account
- **Their textbooks are invisible to software** — legacy encoding fonts (Baamini, SHREE-TAM7) cause every standard PDF extractor to return empty or garbled text

And the crisis is growing. Sri Lanka ranks **4th most climate-affected country globally**. Cyclone Ditwah (late 2025) damaged **1,382 schools** and displaced **550,000 students**. When roads are blocked and schools are closed for months, students need a tool that works **anywhere, on any phone, in airplane mode**.

---

## The Solution

**Nunarivu AI** runs **Google Gemma 4 E2B via LiteRT-LM** entirely on the Android device. No server. No subscription. No internet — ever.

```
Tamil Textbook PDF
  → Android PdfRenderer (2× PNG)        # same as pdf2image below
  → Tesseract OCR (tam+eng, PSM6, OEM3) # same params as this notebook
  → SQLite cache (permanent, per page)   # only OCR'd once
  → Gemma 4 E2B via LiteRT-LM JNI       # same prompt as this notebook
  → Streaming Tamil answer to student
```

---
## Section 1 — Setup

Install Tesseract with Tamil language support and Python dependencies.

In [ ]:
# Install Tesseract OCR with Tamil language pack
!apt-get install -y tesseract-ocr tesseract-ocr-tam tesseract-ocr-eng 2>/dev/null | tail -3

# Python packages
!pip install pytesseract pdf2image Pillow -q

import pytesseract
from PIL import Image
import IPython.display as ipd
import os, textwrap, warnings
warnings.filterwarnings('ignore')

# Verify Tesseract and Tamil language pack
langs = pytesseract.get_languages()
print(f"Tesseract version : {pytesseract.get_tesseract_version()}")
print(f"Available languages: {langs}")
assert 'tam' in langs, "Tamil language pack not installed!"
print("\n✅ Tesseract ready with Tamil support")

---
## Section 2 — The Core Problem: Legacy Fonts Break Every PDF Extractor

Sri Lankan government Tamil textbooks use legacy encoding fonts (**Baamini**, **SHREE-TAM7**, **FMAbhaya**) with WinAnsiEncoding and no Unicode CMap.

Every standard PDF text extractor — PyMuPDF, PDFBox, pdfplumber — returns **zero readable text** from these books. This makes them completely invisible to every existing AI education tool.

In [ ]:
import base64, io

# ── Load the sample textbook page ──────────────────────────────────────────
# The dataset contains a real Sri Lankan Grade 10 Tamil textbook page
# rendered at 2× scale (same as the Android app does via PdfRenderer)

DATASET_PATH = "/kaggle/input/nunarivu-sample-pages"

# Find available page images
page_files = sorted([f for f in os.listdir(DATASET_PATH) if f.endswith('.png')])
print(f"Found {len(page_files)} page images: {page_files}")

PAGE_IMAGE_PATH = os.path.join(DATASET_PATH, page_files[0])
page_img = Image.open(PAGE_IMAGE_PATH)
print(f"Page resolution: {page_img.width} × {page_img.height} px (2× scale for OCR quality)")

# Show the textbook page
print("\n📖 Sri Lankan Tamil Government Textbook Page:")
display_img = page_img.resize((page_img.width // 2, page_img.height // 2), Image.LANCZOS)
display(display_img)

In [ ]:
# Attempt standard PDF text extraction — demonstrate the failure
PDF_PATH = os.path.join(DATASET_PATH, "raw_page.pdf") if os.path.exists(
    os.path.join(DATASET_PATH, "raw_page.pdf")) else None

if PDF_PATH:
    try:
        import fitz  # PyMuPDF
        doc = fitz.open(PDF_PATH)
        raw_text = doc[0].get_text()
        print("PyMuPDF extraction result:")
        print(repr(raw_text[:200]) if raw_text.strip() else repr(raw_text))
        if not raw_text.strip():
            print("\n❌ RESULT: Empty string — PyMuPDF returns ZERO readable text")
            print("   This is the legacy font encoding problem Nunarivu AI solves.")
        else:
            print("\n⚠️  Some text extracted but likely garbled WinAnsi bytes:")
            print(raw_text[:300])
    except ImportError:
        print("PyMuPDF not available in this environment.")
        print("Known result: legacy font PDFs return empty string from get_text()")
else:
    print("📋 No PDF provided in dataset (PNG images only — showing the OCR approach directly)")
    print()
    print("Known behaviour with legacy-font Tamil PDFs:")
    print("  PyMuPDF  doc[0].get_text()  → '' (empty)")
    print("  pdfplumber page.extract_text() → None")
    print("  PDFBox   getText()           → garbled WinAnsi bytes")
    print()
    print("These books use Baamini/SHREE-TAM7 fonts with no Unicode CMap.")
    print("No standard extractor can read them. This is why Tesseract OCR is needed.")

---
## Section 3 — OCR Pipeline: Tesseract Extracts Real Tamil Text

The app uses **exactly these parameters** (from `lib/services/ocr_service.dart`):
- Language: `tam+eng` (Tamil + English)
- OEM 3: LSTM neural net engine (best accuracy)
- PSM 6: Assume a uniform block of text
- Resolution: 2× page scale for sub-pixel Tamil glyph detail

Results are cached permanently in SQLite — each page is OCR'd only once, then instant on every subsequent question.

In [ ]:
# ── Tesseract OCR — exact same parameters as ocr_service.dart ──────────────
print("Running Tesseract OCR (tam+eng, OEM 3, PSM 6)...")
print("This mirrors the Android app's OcrService exactly.\n")

ocr_text = pytesseract.image_to_string(
    page_img,
    lang='tam+eng',
    config='--oem 3 --psm 6'
)

# Clean up (same as app's trim logic)
ocr_text = ocr_text.strip()

print("✅ OCR complete. Extracted text:")
print("─" * 60)
print(ocr_text)
print("─" * 60)
print(f"\nTotal characters extracted: {len(ocr_text)}")
print(f"Tamil characters detected : {sum(1 for c in ocr_text if '\u0B80' <= c <= '\u0BFF')}")

---
## Section 4 — Load Gemma 4

The Android app runs **Gemma 4 E2B** via LiteRT-LM on the device GPU (Adreno 710, OpenCL).  
Here we load the same model via HuggingFace `transformers` on Kaggle's T4 GPU.

The model is accessed from Kaggle's model hub — add it via **"Add Model"** in the notebook sidebar.

> **To add the model**: In Kaggle notebook editor → right panel → **"Add Model"** → search **"gemma-4"** → select the `gemma-4-it` variant → Add.

In [ ]:
import torch
print(f"GPU available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# ── Locate Gemma 4 model (Kaggle model hub path) ───────────────────────────
# After adding the model in Kaggle's sidebar, it appears at one of these paths:
POSSIBLE_PATHS = [
    "/kaggle/input/gemma-4/transformers/gemma-4-it/1",
    "/kaggle/input/gemma-4/transformers/gemma-4-it-4b/1",
    "/kaggle/input/gemma2/transformers/gemma-4-it/1",
    "/kaggle/input/gemma-4-it",
]

MODEL_PATH = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        MODEL_PATH = p
        print(f"Found model at: {p}")
        break

if MODEL_PATH is None:
    # Fallback: try HuggingFace (requires internet + HF token in Kaggle secrets)
    print("Kaggle model not found — trying HuggingFace...")
    MODEL_PATH = "google/gemma-4-it"

print(f"\nLoading {MODEL_PATH} ...")

# 4-bit quantisation so it fits in T4 VRAM (16 GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
print("\n✅ Gemma 4 loaded and ready")

---
## Section 5 — Tamil Q&A: Gemma 4 Answers from the Textbook Page

This is the **exact prompt** the Android app uses (from `lib/screens/pdf_viewer_screen.dart`, method `_buildPrompt()`).  
The model is instructed to answer **only from the OCR'd page text** — no outside knowledge, no hallucination.

In [ ]:
# ── Exact prompt from pdf_viewer_screen.dart _buildPrompt() ────────────────
def build_prompt(
    page_text: str,
    question: str,
    page_num: int = 1,
    title: str = "Grade 10 Science Textbook",
    lang: str = "Tamil",
) -> str:
    return (
        f'You are a teacher answering a student\'s question '
        f'about their textbook "{title}", page {page_num}.\n'
        f'Answer STRICTLY using ONLY the page text below. '
        f'Do NOT use any outside knowledge.\n'
        f'Do NOT add motivational phrases.\n'
        f'If the answer cannot be found in the page text, reply ONLY with:\n'
        f'"இந்தப் பக்கத்தில் இந்தத் தகவல் இல்லை · This information is not on this page."\n\n'
        f'PAGE TEXT (page {page_num}):\n"""\n{page_text}\n"""\n\n'
        f'Question: {question}\n\n'
        f'Answer in {lang}:'
    )


def ask_gemma(question: str, page_text: str, max_new_tokens: int = 350) -> str:
    prompt = build_prompt(page_text, question)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("ask_gemma() helper ready ✅")

In [ ]:
# ── Demo Question 1 — about something on the page ─────────────────────────
QUESTION_1 = "இந்தப் பக்கத்தில் முக்கியமான கருத்து என்ன?"  # What is the main concept on this page?

print(f"Question: {QUESTION_1}")
print("─" * 60)
print("Gemma 4 Answer (from textbook page only):")
print()

answer_1 = ask_gemma(QUESTION_1, ocr_text)
print(answer_1)

In [ ]:
# ── Demo Question 2 — something NOT on the page (safety test) ─────────────
# The app hard-blocks hallucination: if the answer isn't on the page,
# Gemma 4 must reply with the exact Tamil refusal message.

QUESTION_2 = "ஐன்ஸ்டீனின் பிறந்த ஆண்டு என்ன?"  # What year was Einstein born? (not in any textbook page)

print(f"Question: {QUESTION_2}")
print("(This is NOT in the textbook page — testing the hallucination block)")
print("─" * 60)
print("Gemma 4 Answer:")
print()

answer_2 = ask_gemma(QUESTION_2, ocr_text)
print(answer_2)
print()

refusal = "இந்தப் பக்கத்தில் இந்தத் தகவல் இல்லை"
if refusal in answer_2:
    print("✅ Hallucination block working — model refused to guess from outside knowledge")
else:
    print("⚠️  Model answered anyway — adjust prompt or temperature for stricter grounding")

---
## Section 6 — ✏️ Try Your Own Question

Change `YOUR_QUESTION` below and **run this cell**. Ask anything — in Tamil or English.  
Gemma 4 will answer only from the textbook page shown above.

In [ ]:
# ✏️ CHANGE THIS LINE — ask any question about the textbook page above
YOUR_QUESTION = "இந்தப் பாடத்தில் என்ன கற்றுக்கொள்ள வேண்டும்?"
# Translation: "What should I learn from this lesson?"

# ─────────────────────────────────────────────────────────────────────────
print(f"Your question: {YOUR_QUESTION}")
print("─" * 60)
your_answer = ask_gemma(YOUR_QUESTION, ocr_text)
print(your_answer)

In [ ]:
# Try a second page if available
if len(page_files) > 1:
    print("📖 Running OCR on page 2...")
    page2_img = Image.open(os.path.join(DATASET_PATH, page_files[1]))
    display(page2_img.resize((page2_img.width // 2, page2_img.height // 2), Image.LANCZOS))

    ocr_text_2 = pytesseract.image_to_string(page2_img, lang='tam+eng', config='--oem 3 --psm 6').strip()
    print(f"OCR extracted {len(ocr_text_2)} characters from page 2\n")

    # ✏️ CHANGE THIS — question about page 2
    Q2 = "இந்தப் பக்கத்தில் என்ன விளக்கப்படுகிறது?"
    print(f"Question: {Q2}")
    print("─" * 60)
    print(ask_gemma(Q2, ocr_text_2))
else:
    print("Only one page image in dataset. Add page_002.png to the dataset for a multi-page demo.")

---
## Section 7 — How This Maps to the Android App

Every step in this notebook corresponds exactly to code running on the student's phone:

| This Notebook (Kaggle T4 GPU) | Android App (Adreno 710 GPU) |
|-------------------------------|------------------------------|
| `pdf2image` / PNG input | Android `PdfRenderer` at 2× scale |
| `pytesseract` (PSM 6, OEM 3) | `flutter_tesseract_ocr` NDK plugin |
| `transformers` model loading | LiteRT-LM C++ engine via JNI |
| Python `build_prompt()` | Dart `_buildPrompt()` in `pdf_viewer_screen.dart` |
| `model.generate()` blocking call | `EventChannel` streaming tokens to Flutter UI |
| Console output | Real-time Tamil text + TTS read-aloud |

### The Android Streaming Bridge (Kotlin)

```kotlin
// InferencePlugin.kt — streams Gemma 4 tokens to Flutter via EventChannel
streamJob = scope.launch {
    conversation = createConversationWithRecovery(engine)
    val flow = conversation.sendMessageAsync(prompt)
    flow.collect { chunk ->
        withContext(Dispatchers.Main) { events.success(chunk.toString()) }
    }
    withContext(Dispatchers.Main) { events.endOfStream() }
}
```

### The OCR Cache (Dart)

```dart
// ocr_service.dart — each page OCR'd once, cached permanently in SQLite
text = await FlutterTesseractOcr.extractText(
    imgPath,
    language: 'tam+eng',
    args: { 'psm': '6', 'oem': '3' },
);
// Stored with chunk_index = -1 in pdf_chunks table
```

### Why LiteRT-LM on the Phone?

- **`transformers`** (this notebook) requires internet + cloud GPU — not available to rural Tamil students
- **LiteRT-LM** runs the same Gemma 4 model as a single `.litertlm` file on the Adreno 710 GPU
- Students side-load the 2.4 GB model once via USB, then it's theirs forever
- First token in ~8–12 seconds; streaming at ~4–6 tokens/sec on Adreno 710

---

## App Features Beyond This Notebook

| Feature | Technology |
|---------|------------|
| **Tamil Voice Q&A** | Android offline `ta-IN` STT — no internet |
| **Read-aloud answers** | `flutter_tts` — offline TTS |
| **161 textbooks** | Grade 10 + 11 full curriculum, auto-imported |
| **Multi-student profiles** | Siblings share one phone — separate history/badges |
| **Study reminders** | Daily Tamil notifications — "படிக்க நேரம்!" |
| **Achievement badges** | Unlocked by question count + study-day streaks |
| **100% offline** | Works in airplane mode — zero data ever leaves the phone |

---

## Try the App

- **APK**: https://github.com/sanjeevakumar610/Nunarivu-AI/releases/download/v1.0.0/app-release.apk
- **Source**: https://github.com/sanjeevakumar610/Nunarivu-AI
- **Model**: https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm

Install APK → download the Gemma 4 model → copy to device → turn on **Airplane Mode** → ask in Tamil. Everything works.

---

> *Gemma is a trademark of Google LLC. Nunarivu AI is an independent open-source project not affiliated with or endorsed by Google.*
>
> *🔒 Zero data leaves the student's device. Ever.*